In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas

In [ ]:
import json
import os
import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig
from collections import defaultdict
from google.colab import drive
import logging

## Model Setup

In [ ]:
# --- Setup Logging ---
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

CONFIG = {
    "model_name": "microsoft/deberta-v3-large",
    "max_length": 512,
    "batch_size": 8,
    "seed": 42,
    "EXCLUDED_RELATIONS": {'no_relation', 'unanswerable', 'per:alternate_names', 'per:place_of_birth'},
    "use_fp16": True

}

# --- Set Seed for Reproducibility ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])

In [ ]:
drive.mount('/content/drive')

## Loading Model

In [ ]:
# IMPORTANT: Define the paths to your artifacts and raw data
DRIVE_MODEL_DIR = "/path/to/data"
MODEL_WEIGHTS_PATH = os.path.join(DRIVE_MODEL_DIR, "deberta-large_model.pth")
LABEL_MAP_PATH = os.path.join(DRIVE_MODEL_DIR, "label_map.json")

# UPDATE THIS PATH to your raw data file (e.g., test.json)
RAW_DATA_PATH = "/path/to/test.json"

print(f"Loading model from: {MODEL_WEIGHTS_PATH}")
print(f"Loading data from: {RAW_DATA_PATH}")

In [ ]:
try:
    with open(LABEL_MAP_PATH, 'r') as f:
        label_map = json.load(f)
        label2id = label_map['label2id']
        id2label = label_map['id2label']
except FileNotFoundError:
    logger.error(f"Label map not found at {LABEL_MAP_PATH}. Check your path!")
    raise

NUM_LABELS = len(label2id)

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

special_tokens = {
    's_start': '[SS]', 's_end': '[/SS]',
    'o_start': '[OS]', 'o_end': '[/OS]'
}
tokenizer.add_special_tokens({'additional_special_tokens': list(special_tokens.values())})
logger.info(f"Loaded tokenizer with {len(tokenizer)} tokens.")


class REDataset(Dataset):
    def __init__(self, samples, tokenizer, label2id, max_len=512):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        text = item['text']

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(0, dtype=torch.long)
        }



### Model Architecture


In [ ]:
class UniversalREModel(nn.Module):
    def __init__(self, model_name, num_labels, tokenizer):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        # CRITICAL: Resize to match the saved weights!
        self.bert.resize_token_embeddings(len(tokenizer))

        self.dropout = nn.Dropout(0.3) # Use your trained dropout rate

        self.s_start_id = tokenizer.convert_tokens_to_ids('[SS]')
        self.o_start_id = tokenizer.convert_tokens_to_ids('[OS]')

        self.classifier = nn.Linear(self.config.hidden_size * 2, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state

        batch_size, seq_len, hidden_size = sequence_output.shape

        s_emb = torch.zeros(batch_size, hidden_size).to(input_ids.device)
        o_emb = torch.zeros(batch_size, hidden_size).to(input_ids.device)

        for i in range(batch_size):
            s_idx = (input_ids[i] == self.s_start_id).nonzero(as_tuple=True)[0]
            o_idx = (input_ids[i] == self.o_start_id).nonzero(as_tuple=True)[0]

            if s_idx.numel() > 0:
                s_emb[i] = sequence_output[i, s_idx[0], :]
            else:
                s_emb[i] = sequence_output[i, 0, :]

            if o_idx.numel() > 0:
                o_emb[i] = sequence_output[i, o_idx[0], :]
            else:
                o_emb[i] = sequence_output[i, 0, :]

        combined = torch.cat([s_emb, o_emb], dim=1)
        combined = self.dropout(combined)
        logits = self.classifier(combined)

        return {"logits": logits}

### Data processing

In [ ]:
def process_raw_data_for_inference(file_path, tokenizer, special_tokens):
    """Loads raw DialogRE data and prepares a list of inference samples using robust entity matching."""

    if not os.path.exists(file_path):
        logger.error(f"Raw data file not found: {file_path}")
        return []

    logger.info(f"Attempting to load data from: {file_path}")
    with open(file_path, 'r') as f:
        raw_data = json.load(f)

    logger.info(f"Loaded {len(raw_data)} conversations from raw data.")
    if not raw_data:
        logger.warning("Raw data is empty after loading.")
        return []

    inference_samples = []

    for conv_id, conv in enumerate(raw_data):
        if conv_id < 5:
            logger.debug(f"Processing conversation ID: {conv_id}")

        dialogue = conv[0]
        relations = conv[1]

        dialogue_text_list = dialogue

        for rel_idx, rel in enumerate(relations):
            if conv_id < 5 and rel_idx < 5:
                logger.debug(f"  Processing relation {rel_idx} for conv_id {conv_id}: x={rel['x']}, y={rel['y']}, r={rel['r']}")

            x_text = rel['x']
            y_text = rel['y']

            x_norm = x_text.lower().strip()

            for r_label in rel['r']:
                if r_label in label2id:
                    if conv_id < 5 and rel_idx < 5:
                        logger.debug(f"    Relation '{r_label}' found in label2id. Proceeding.")
                    else:
                        if conv_id == 5 and rel_idx == 0:
                            logger.debug(f"    (Suppressing debug for conv_id > 4 or rel_idx > 4)")

                    utterance_idx = -1
                    for i, utt in enumerate(dialogue_text_list):
                        if x_norm in utt.lower(): # Match normalized subject against normalized utterance
                            utterance_idx = i
                            break

                    if utterance_idx == -1:
                        if conv_id < 5 and rel_idx < 5:
                            logger.warning(f"    Subject '{x_text}' (normalized '{x_norm}') not found in dialogue for conv_id {conv_id}, relation {rel_idx}. Skipping.")
                        continue

                    # 2. Insert Markers for Model Input (Using original casing for tokenization)
                    full_text = " ".join(dialogue_text_list)
                    s_mark = f"{special_tokens['s_start']} {x_text} {special_tokens['s_end']}"
                    o_mark = f"{special_tokens['o_start']} {y_text} {special_tokens['o_end']}"


                    processed_text = full_text.replace(x_text, s_mark, 1).replace(y_text, o_mark, 1)

                    inference_samples.append({
                        'conv_id': conv_id,
                        'text': processed_text,
                        'subject': x_text,
                        'object': y_text,
                        'target_idx': utterance_idx,
                        'dialogue_list': dialogue_text_list
                    })
                else:
                    if conv_id < 5 and rel_idx < 5:
                        logger.debug(f"    Relation '{r_label}' NOT found in label2id. Skipping relation.")

    logger.info(f"Prepared {len(inference_samples)} samples for prediction.")
    return inference_samples

# --- Canonical ID Generation Function ---
def create_canonical_id(conv_id, entity_name):
    """
    Generates a unique KG ID based on whether the entity is a generic speaker
    or a known character name (Global for names, Conversation-specific for generic).

    Generic speakers are now formatted as: [conv_id]_Spkr_[number] (e.g., '0_Spkr_1').
    """
    entity_lower = entity_name.lower()

    # Check 1: If it contains 'speaker' or starts with 's' followed by a number (e.g., "Speaker 1", "S1")
    if 'speaker' in entity_lower or (entity_lower.startswith('s') and len(entity_name.replace('.', '').strip()) <= 3):


        new_name = str(conv_id) + '_'

        processed_entity = (
            entity_name.replace('Speaker', 'Spkr')
                       .replace('speaker', 'Spkr')
                       .replace('S.', 'Spkr')
        )

        # Append the processed entity name, replacing spaces/dots with underscores
        new_name += processed_entity.replace(' ', '_').replace('.', '')

        return new_name

    else:
        # Named Entity: Use the name as the global ID
        return entity_name

## Main Exetusion functuon

In [ ]:
# --- Main Inference Function ---
def run_inference():
    # Load raw data and prepare samples
    inference_data = process_raw_data_for_inference(RAW_DATA_PATH, tokenizer, special_tokens)

    # Robustness Check
    if not inference_data:
        logger.warning("No data samples were prepared for inference. Returning empty DataFrame.")
        return pd.DataFrame()

    # 1. Initialize Model and Load Weights
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = UniversalREModel(CONFIG['model_name'], NUM_LABELS, tokenizer)
    model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
    model.to(device)
    model.eval()
    logger.info("Model loaded successfully.")

    # 2. Create DataLoader
    inference_dataset = REDataset(inference_data, tokenizer, label2id, CONFIG['max_length'])
    inference_loader = DataLoader(inference_dataset, batch_size=CONFIG['batch_size'], num_workers=2)

    all_results = []

    # 3. Run Predictions
    logger.info("Starting prediction...")
    for batch_idx, batch in enumerate(inference_loader):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids, mask)

        logits = outputs['logits']
        predictions = torch.argmax(logits, dim=-1).cpu().numpy()

        # 4. Store Results along with Original Data
        start_idx = batch_idx * CONFIG['batch_size']

        for i, pred_id in enumerate(predictions):
            sample = inference_data[start_idx + i]

            # FIX: Convert the predicted integer ID to a string to match the id2label dictionary keys
            predicted_relation = id2label[str(pred_id)]

            # Filter for PER-PER relations
            if predicted_relation.startswith('per:'):
                all_results.append({
                    'conv_id': sample['conv_id'],
                    'subject': sample['subject'],
                    'object': sample['object'],
                    'relation': predicted_relation,
                    'dialogue_list': sample['dialogue_list'],
                    'target_idx': sample['target_idx']
                })

    logger.info(f"Finished prediction. Found {len(all_results)} PER-PER relations.")

    # 5. Format Output DataFrame with Canonical IDs
    final_df_rows = []
    for result in all_results:
        d_list = result['dialogue_list']
        target_idx = int(result['target_idx'])

        # --- KG ID GENERATION ---
        subject_kg_id = create_canonical_id(result['conv_id'], result['subject'])
        object_kg_id = create_canonical_id(result['conv_id'], result['object'])

        # Extract surrounding context (handle boundaries)
        target_turn = d_list[target_idx] if 0 <= target_idx < len(d_list) else ""
        minus_1_turn = d_list[target_idx - 1] if target_idx > 0 else "N/A"
        plus_1_turn = d_list[target_idx + 1] if target_idx < len(d_list) - 1 else "N/A"


        context_sentence = (
            f"(-1 Turn: {minus_1_turn}) "
            f"(Target Turn: {target_turn}) "
            f"(+1 Turn: {plus_1_turn})"
        )

        final_df_rows.append({
            'conv_id': result['conv_id'],
            'sentence': context_sentence,
            'subject_id': subject_kg_id,
            'object_id': object_kg_id,
            'relation': result['relation'],
            'subject_name': result['subject'],
            'object_name': result['object']
        })

    # Create the DataFrame
    emotion_attribution_df = pd.DataFrame(final_df_rows)

    return emotion_attribution_df

# Execute the inference pipeline
try:
    final_emotion_df = run_inference()

    if not final_emotion_df.empty:
        print("\n--- Final DataFrame for Emotion Attribution ---")
        print(final_emotion_df.head(10))
        print(f"\nTotal relations extracted: {len(final_emotion_df)}")

        # Save the DataFrame to Google Drive
        SAVE_DF_PATH = os.path.join(DRIVE_MODEL_DIR, "inferred_per_relations_for_emotion.csv")
        final_emotion_df.to_csv(SAVE_DF_PATH, index=False)
        print(f"DataFrame saved to: {SAVE_DF_PATH}")
    else:
        print("\n--- No Relations Found or Data Preparation Failed. DataFrame is Empty. ---")

except Exception as e:
    logger.error(f"An error occurred during inference: {type(e).__name__}: {e}")

## Filtering non-per-relations

In [ ]:
relations_to_filter = [
    'per:age',
    'per:visited_place',
    'per:works',
    'per:title',
    'per:place_of_work',
    'per:place_of_residence',
    'per:employee_or_member_of',
    'per:client',
    'per:alumni',
    'per:dates'
]

if 'relation' in final_emotion_df.columns:
    filtered_emotion_df = final_emotion_df[~final_emotion_df['relation'].isin(relations_to_filter)]
    display(filtered_emotion_df.head(5))
    print(f"\nTotal relations after filtering: {len(filtered_emotion_df)}")
else:
    print("Error: 'final_emotion_df' is empty or does not have a 'relation' column. Please ensure your RAW_DATA_PATH is correct and data is loaded.")
    filtered_emotion_df = pd.DataFrame() # Initialize as empty to prevent further errors

In [ ]:
# Remove duplicate rows
filtered_emotion_df = filtered_emotion_df.drop_duplicates()
print(f"Total relations after removing duplicates: {len(filtered_emotion_df)}")

filtered_emotion_df = filtered_emotion_df.dropna(axis=1, how='all')
print(f"DataFrame columns after removing empty columns: {filtered_emotion_df.columns.tolist()}")

display(filtered_emotion_df.head(5))
print(f"\nFinal total relations after all filtering and cleaning: {len(filtered_emotion_df)}")

In [ ]:
SAVE_FILTERED_DF_PATH = os.path.join(DRIVE_MODEL_DIR, "filtered_per_relations.csv")
filtered_emotion_df.to_csv(SAVE_FILTERED_DF_PATH, index=False)
print(f"Filtered DataFrame saved to: {SAVE_FILTERED_DF_PATH}")

In [ ]:
import pandas as pd
import os

#
file_path = SAVE_FILTERED_DF_PATH

# Load the dataset
df = pd.read_csv(file_path)

friends_df = df[df['relation'] == 'per:friends'].copy()

num_friends_relations = len(friends_df)
if num_friends_relations == 0:
    print("No 'per:friends' relations found in the dataset.")
    random_friends = pd.DataFrame(columns=['subject_id', 'object_id', 'relation', 'subject_name', 'object_name'])
else:
    n_samples = min(10, num_friends_relations)
    random_friends = friends_df.sample(n=n_samples)[['subject_id', 'object_id', 'relation', 'subject_name', 'object_name']]

print("10 Random 'per:friends' relations:")
print(random_friends)

# Save the resulting DataFrame to a new CSV file
output_path = os.path.join(DRIVE_MODEL_DIR, '10_random_friends_relations.csv') # Save to Drive as well
random_friends.to_csv(output_path, index=False)
print(f"\nSaved the random relations to {output_path}")

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv('/path/to/filtered_per_relations.csv')

df['context_length'] = df['sentence'].str.len()

# 2. Stratified Sampling: 10 samples per relation, then fill the rest
target_total = 150
unique_relations = df['relation'].unique()
samples_per_rel = target_total // len(unique_relations)

selected_indices = []
for rel in unique_relations:
    rel_group = df[df['relation'] == rel].sort_values(by='context_length', ascending=False)
    n = min(len(rel_group), samples_per_rel)
    selected_indices.extend(rel_group.head(n).index.tolist())

# Fill up to 150 with remaining longest sentences
if len(selected_indices) < target_total:
    remaining = df.drop(selected_indices).sort_values(by='context_length', ascending=False)
    selected_indices.extend(remaining.head(target_total - len(selected_indices)).index.tolist())

df_150_pygents = df.loc[selected_indices].copy()
df_150_pygents['manual_label'] = ""
df_150_pygents.to_csv('pygents_samples_150.csv', index=False)

In [ ]:
SAVE_PYGENTS_DF_PATH = os.path.join(DRIVE_MODEL_DIR, "pygents_samples_150.csv")

# Save the DataFrame to CSV
df_150_pygents.to_csv(SAVE_PYGENTS_DF_PATH, index=False)
print(f"DataFrame 'df_150_pygents' saved to: {SAVE_PYGENTS_DF_PATH}")